In [1]:
import os

if os.getcwd() == '/home/dcor/niskhizov':
    os.chdir('//home/dcor/niskhizov/PhysicalAdverserialProj/')
    on_remote = True
else:
    on_remote = False

In [2]:
from comet_ml import start, ExistingExperiment
from comet_ml.integration.pytorch import log_model

experiment = start(
  api_key="Bg5eubpUjdi2CCiiA5OSoltfw",
  project_name="physicaladvproj-ablation",
  workspace="dannynis"
)

experiment_key = experiment.get_key()
print(f"Experiment key: {experiment_key}")

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/dannynis/physicaladvproj-ablation/58c779acb0414dbcbb1f2389db28a813



Experiment key: 58c779acb0414dbcbb1f2389db28a813


COMET INFO: Couldn't find a Git repository in '/home/dcor/niskhizov/PhysicalAdverserialProj' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


In [3]:
%matplotlib inline

In [4]:
import torch
import torch.nn as nn
import torchvision.models as models

import glob
import matplotlib.pyplot as plt
import os
import cv2
import numpy as np
import time
from tqdm import tqdm
import copy
from IPython.display import display, Image, clear_output
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import datetime
import kornia
import torchvision
import pickle as pkl

tt = torchvision.transforms.ToTensor()

import cv2
import cv2.aruco as aruco

from consts import border_size, displayed_aruco_code, marker_size

# Import classifier components
from classfier_ensemble import predict_raw, orig_clases, model, weights
from classfier_ensemble import predict_raw as predict_raw_dev
from classfier_ensemble import predict_raw as predict_raw_test
from functools import partial

from classfier_test import model as model_test
from classfier import model as model_dev

model_name = model if type(model) == str else model.__class__.__name__

print(f'For training using classifier model: {model.__class__.__name__}')
print('###############################')
print(f'For Dev using classifier model: {model_dev.__class__.__name__}')
print('###############################')
print(f'For Test using classifier model: {model_test.__class__.__name__}')

if 'weight_dict' in predict_raw.__code__.co_varnames:
    classfiers_weights_dict = {
        'inception': 0.25,
        'resnet': 0.25,
        'vgg': 0.25,
        'vit': 0.25,
        'dino': 0.0
    }
    experiment.log_parameters(classfiers_weights_dict)
    predict_raw = partial(predict_raw, weights_dict=classfiers_weights_dict)

print('#################################')
print(f'ORIG CLASES {orig_clases}')

# Load photometric calibration
photometric_calibrations_dir = './photometric_calibrations'
photometric_calibration_files = glob.glob(os.path.join(photometric_calibrations_dir, 'photometric_calibration_*.pkl'))
photometric_calibration_files.sort(key=os.path.getmtime, reverse=True)
photometric_calibration_path = photometric_calibration_files[0]
print(f'############## Loading photometric calibration data from {photometric_calibration_path} ##############')

with open(photometric_calibration_path, "rb") as f:
    data = pkl.load(f)

height = data['height']
width = data['width']

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ArUco setup
aruco_dict_type = cv2.aruco.DICT_4X4_50
marker_length = 0.05
aruco_dict = cv2.aruco.getPredefinedDictionary(aruco_dict_type)
marker_id = displayed_aruco_code
marker_size = marker_size
marker_image = cv2.aruco.generateImageMarker(aruco_dict, marker_id, marker_size)
aruco_dict = aruco.getPredefinedDictionary(aruco_dict_type)
parameters = aruco.DetectorParameters()
detector = aruco.ArucoDetector(aruco_dict, parameters)

# Load Stable Diffusion VAE
from diffusers import StableDiffusionPipeline

def decode_latents_grad(latents):
    latents = 1 / 0.18215 * latents
    imgs = vae.decode(latents).sample
    imgs = (imgs / 2 + 0.5).clamp(0, 1)
    return imgs

def decode_latents(latents):
    with torch.no_grad():
        with torch.amp.autocast(device):
            latents = 1 / 0.18215 * latents
            with torch.no_grad():
                imgs = vae.decode(latents).sample
            imgs = (imgs / 2 + 0.5).clamp(0, 1)
    return imgs

def encode_imgs(imgs):
    with torch.no_grad():
        with torch.amp.autocast(device):
            imgs = 2 * imgs - 1
            posterior = vae.encode(imgs).latent_dist
            latents = posterior.sample() * 0.18215
    return latents

class framesDataset(Dataset):
    def __init__(self, frames, Hs):
        self.frames = frames
        self.Hs = Hs

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, idx):
        frame = self.frames[idx]
        H = self.Hs[idx]
        frame_tensor = tt(frame)
        return frame_tensor, H.astype(np.float32)

valid_frames = None

def warp(decoded_latents, H_t):
    dst_img_shape = valid_frames[0].shape[:2]
    warped_imgs = []
    for decoded_latent in decoded_latents:
        img = decoded_latent.unsqueeze(0).float().repeat(H_t.shape[0], 1, 1, 1)
        w = kornia.geometry.transform.warp_perspective(img, H_t, dst_img_shape)
        warped_imgs.append(w)
    return torch.stack(warped_imgs, dim=0)

vae = None

os.makedirs('./results', exist_ok=True)
os.makedirs('./results/ablation', exist_ok=True)
curr_without_sec = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
curr_without_sec = curr_without_sec.replace(" ", "_").replace(":", "_")

pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")
vae = pipe.vae.to(device).eval()

try:
    first_class_idx = orig_clases[0].item()
    first_class_name = weights.meta['categories'][first_class_idx]
    experiment_name = f"ablation_{model_name}_{curr_without_sec}_{first_class_name}"
    experiment.set_name(experiment_name)
    print(f"Experiment name set to: {experiment_name}")
except Exception as e:
    print(f"Could not set experiment name: {e}")

categories = weights.meta['categories']

Loading ensemble models...


Using cache found in /home/dcor/niskhizov/cache/hub/facebookresearch_dinov2_main
/home/dcor/niskhizov/cache/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/dcor/niskhizov/cache/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/dcor/niskhizov/cache/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")
INFO:dinov2:using MLP layer as FFN


✓ All models loaded successfully!
  - Inception V3
  - ResNet18
  - VGG16
  - ViT-B/16
  - DINOv2

✓ Ensemble classifier ready!
  Main function: predict_raw(image)
  Alternatives: predict_raw_weighted(image, weights_dict)
               predict_raw_per_model(image)
               ensemble_predict(image)
For training using classifier model: str
###############################
For Dev using classifier model: Inception3
###############################
For Test using classifier model: EfficientNet
#################################
ORIG CLASES tensor([636, 414, 748], device='cuda:0')
############## Loading photometric calibration data from ./photometric_calibrations/photometric_calibration_20260117_202634.pkl ##############


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Experiment name set to: ablation_ensamble_classifier_2026-01-17_21_28_mailbag


In [5]:
# Print configuration
print(f'Using border size {border_size} for ArUco detection')

Using border size 2 for ArUco detection


In [6]:
def find_border_drop_point(gray, c):
    sub = np.subtract
    add = np.add
    borders_drop_points = []
    for idx, operators in enumerate(([sub,sub],[add,sub],[add,add],[sub,add])):
        margin = 1
        a, b = int(c[idx][0]), int(c[idx][1])
        diag_idxs = np.arange(5)
        nca = operators[0](a, diag_idxs)
        ncb = operators[1](b, diag_idxs)
        nc = np.stack([nca, ncb], axis=1)
        diag_line_vals = gray[nc[:, 1], nc[:, 0]].astype(np.float32)
        diag_line_vals_diff = np.diff(diag_line_vals)
        if np.all(diag_line_vals_diff >= 0):
            borders_drop_points.append((nca[0], ncb[0]))
            continue
        diag_line_vals_diff_first_neg = min(np.where(diag_line_vals_diff < 0)[0][0] + margin, len(diag_line_vals_diff)-1)
        new_a = nca[diag_line_vals_diff_first_neg]
        new_b = ncb[diag_line_vals_diff_first_neg]
        borders_drop_points.append((new_a, new_b))
    return np.array(borders_drop_points)

caps_dir = 'captures_frames_multiview'
ls = os.listdir(f'./{caps_dir}')
captures = [f for f in ls if f.startswith('captures_frames_multiview_')]
captures = sorted(captures, key=lambda x: int(x.split('_')[-1]))
cap_dir = f'./{caps_dir}/{captures[-1]}'

In [7]:
print(f"Loading frames from: {cap_dir}")
files_sorted = sorted(glob.glob(f'{cap_dir}/*.png'), key=lambda x: int(x.split('_')[-1].split('.')[0]))
print(f"Total frame files found: {len(files_sorted)}")
frames = [cv2.cvtColor(cv2.imread(file), cv2.COLOR_BGR2RGB) for file in files_sorted]
print(f"Loaded {len(frames)} frames")

Loading frames from: ./captures_frames_multiview/captures_frames_multiview_153
Total frame files found: 439
Loaded 439 frames


In [8]:
# Process frames and calculate homographies (matching original notebook logic)
valid_frames = []
H_list = []

# Use orig_img_corners with border_size like in original notebook
orig_img_corners = np.array([
    [border_size, border_size],
    [width - border_size, border_size],
    [width - border_size, height - border_size],
    [border_size, height - border_size]
], dtype=np.float32)

found_aruco_count = 0
outlier_clases = []

for idx, frame in tqdm(enumerate(frames)):
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    corners, ids, rejected = detector.detectMarkers(gray)
    
    if ids is None:
        continue
    
    found_aruco_count += 1
    
    # Check classifier prediction
    with torch.no_grad():
        pr = predict_raw(tt(frame).cuda().unsqueeze(0))
    
    if displayed_aruco_code in ids and pr.argmax(1) in orig_clases:
        displayed_aruco_code_index = np.where(ids.flatten() == displayed_aruco_code)[0][0]
        c = corners[displayed_aruco_code_index][0]
        
        try:
            unbordred_corners = find_border_drop_point(gray, c)
        except:
            print('unbordering failed')
            continue
        
        H, _ = cv2.findHomography(orig_img_corners, unbordred_corners, cv2.RANSAC)
        
        if H is not None:
            valid_frames.append(frame)
            H_list.append(H)
    else:
        outlier_clases.append(pr.argmax(1))

print(f"Detected ArUco markers in {found_aruco_count} out of {len(frames)} frames.")
print(f"Found {len(valid_frames)} valid frames with ArUco markers and original classes.")

# Shuffle and limit frames like original
random_idx = np.random.permutation(min(len(valid_frames), 5000))
valid_frames = [valid_frames[i] for i in random_idx]
H_list = [H_list[i] for i in random_idx]

print(f"Total valid frames to be used for training: {len(valid_frames)}")

if len(valid_frames) == 0:
    raise ValueError("No valid frames found!")

439it [00:20, 21.20it/s]

Detected ArUco markers in 439 out of 439 frames.
Found 435 valid frames with ArUco markers and original classes.
Total valid frames to be used for training: 435


In [9]:
# Create datasets
train_split = int(len(valid_frames) * 0.7)
val_split = int(len(valid_frames) * 0.85)

train_frames = valid_frames[:train_split]
train_Hs = H_list[:train_split]

val_frames = valid_frames[train_split:val_split]
val_Hs = H_list[train_split:val_split]

test_frames = valid_frames[val_split:]
test_Hs = H_list[val_split:]

train_dataset = framesDataset(train_frames, train_Hs)
val_dataset = framesDataset(val_frames, val_Hs)
test_dataset = framesDataset(test_frames, test_Hs)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=10, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Train: 304, Val: 65, Test: 66


In [10]:
# Augmentation setup
jitter = T.Compose([
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.0),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5))
])

jitter_total_photo = T.Compose([
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.0),
])

jitter_with_hue = T.Compose([
    T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
    T.GaussianBlur(kernel_size=5, sigma=(0.1, 1.0))
])

augmentor_model = data['augmentor'].to(device).eval()

## Ablation Study Configuration

We will run 6 experiments:
1. Latent size 4x4 (no rejuvenation)
2. Latent size 8x8 (no rejuvenation)
3. Latent size 16x16 (no rejuvenation)
4. Latent size 32x32 (no rejuvenation)
5. Latent size 16x16 (WITH rejuvenation)
6. Latent size 16x16 (no rejuvenation, duplicate for fair comparison)

In [11]:
# Define experiment configurations
ABLATION_CONFIGS = [
    {"name": "latent_4x4", "latent_size": 4, "rejuvenate": False},
    {"name": "latent_8x8", "latent_size": 8, "rejuvenate": False},
    {"name": "latent_16x16", "latent_size": 16, "rejuvenate": False},
    {"name": "latent_32x32", "latent_size": 32, "rejuvenate": False},
    {"name": "latent_16x16_with_rejuv", "latent_size": 16, "rejuvenate": True},
]

# Common training parameters
NUM_EPOCHS = 30  # Reduced for ablation study
LATENT_BATCH_SIZE = 10  # Number of patches in each batch
BLEND_RATIO = 1.0

print("Ablation Study Configurations:")
for i, cfg in enumerate(ABLATION_CONFIGS, 1):
    print(f"  {i}. {cfg['name']}: latent_size={cfg['latent_size']}x{cfg['latent_size']}, rejuvenate={cfg['rejuvenate']}")

Ablation Study Configurations:
  1. latent_4x4: latent_size=4x4, rejuvenate=False
  2. latent_8x8: latent_size=8x8, rejuvenate=False
  3. latent_16x16: latent_size=16x16, rejuvenate=False
  4. latent_32x32: latent_size=32x32, rejuvenate=False
  5. latent_16x16_with_rejuv: latent_size=16x16, rejuvenate=True


In [12]:
# Results tracking structure
ablation_results = {
    "configs": [],
    "success_rate_history": {},
    "loss_history": {},
    "best_success_rate": {},
    "epochs_to_threshold": {},  # epochs to reach 50% success rate
    "final_aug_rate": {},  # Final augmented success rate
}

In [13]:
def run_single_experiment(config, num_epochs, train_loader, val_loader, experiment, global_results):
    """Run a single ablation experiment and track results."""
    import pickle
    import gc
    
    exp_name = config["name"]
    latent_size = config["latent_size"]
    to_rejuvenate = config["rejuvenate"]
    
    print(f"\n{'='*60}")
    print(f"🧪 Starting Experiment: {exp_name}")
    print(f"   Latent Size: {latent_size}x{latent_size}")
    print(f"   Rejuvenation: {'Enabled' if to_rejuvenate else 'Disabled'}")
    print(f"{'='*60}\n")
    
    # Initialize latent batch for this experiment
    latent_batch = torch.randn((LATENT_BATCH_SIZE, 4, latent_size, latent_size), device=device) * 0.8
    latent_batch.requires_grad = True
    
    # Resizer for this latent size
    resizer = torchvision.transforms.Resize((height, width))
    
    # Optimizer
    latent_opt = torch.optim.Adam([latent_batch], lr=0.1)
    
    # Training state
    orig_clases_np = orig_clases.cpu().numpy()
    num_patches = latent_batch.shape[0]
    
    # Tracking for this experiment
    losses = []
    success_rates = []
    aug_success_rates = []
    best_success_rate = 0
    best_loss = float('inf')
    best_latent = None
    best_patch_idx = None
    training_stopped = False
    aug_weight = 0.9  # Starts at 0.9, will be incrementally increased
    epochs_to_50_percent = None
    
    target_classes = torch.tensor([], device=device)
    
    # Define augmentor as a function that uses the current aug_weight
    def get_augmentor():
        return lambda x: augmentor_model(x).to(device) * aug_weight + x * (1-aug_weight)
    augmentor = get_augmentor()
    
    # Log experiment start
    experiment.log_parameters({
        f"{exp_name}_latent_size": latent_size,
        f"{exp_name}_rejuvenate": to_rejuvenate,
        f"{exp_name}_num_epochs": num_epochs,
    })
    
    for epoch in range(num_epochs):
        epoch_losses = []
        epoch_success_rates = []
        patch_success_history = {i: [] for i in range(num_patches)}
        patch_augmented_success_history = {i: [] for i in range(num_patches)}
        
        # Clear GPU cache at start of each epoch
        torch.cuda.empty_cache()
        
        # Training loop
        for batch_idx, (frames_batch, H_t_batch) in tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            latent_opt.zero_grad()
            
            frames_batch = frames_batch.to(device)
            H_t_batch = H_t_batch.to(device)
            
            # Generate adversarial patch from latent
            adv_patch = resizer(decode_latents_grad(latent_batch).float())
            
            # Apply augmentations
            if torch.rand(1).item() > 0.3:
                adv_patch_aug = jitter(adv_patch)
            else:
                adv_patch_aug = adv_patch
                
            if torch.rand(1).item() > 0.3:
                adv_patch_aug = torch.stack([augmentor(x).to(device) for x in adv_patch_aug])
            
            # Warp and blend
            w_mask = warp(adv_patch_aug * 0 + 1, H_t_batch)
            w_patch = warp(adv_patch_aug, H_t_batch)
            blended_frames = ((w_mask != 0) * -BLEND_RATIO + 1) * frames_batch + w_patch * BLEND_RATIO
            # blended_frames shape: [num_patches, batch_size, 3, H, W] - need to reshape to [N, 3, H, W]
            blended_frames = blended_frames.view(-1, 3, blended_frames.shape[-2], blended_frames.shape[-1])
            
            if torch.rand(1).item() > 0.3:
                blended_frames = jitter_total_photo(blended_frames)
            
            batch_frames = blended_frames
            
            # Get predictions
            with torch.autocast(device_type=device):
                logits = predict_raw(batch_frames)
                if (logits != logits).any():
                    raise ValueError("NaN in logits")
                probs = torch.softmax(logits, dim=1)
            
            # Loss calculation
            orig_class_probs = probs[:, orig_clases]
            orig_loss = 5.0 * torch.log(orig_class_probs.sum(dim=1) + 1e-10).mean()
            
            if target_classes.numel() > 0:
                target_probs = probs[:, target_classes]
                target_loss = -3.0 * torch.log(target_probs.max(dim=1)[0] + 1e-10).mean()
            else:
                target_loss = 0
            
            total_loss = orig_loss + target_loss
            
            # Backward pass
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_([latent_batch], max_norm=3.0)
            latent_opt.step()
            
            epoch_losses.append(total_loss.item())
            
            # Evaluation
            with torch.no_grad():
                predictions = logits.argmax(dim=1)
                successful_attacks = sum(pred.item() not in orig_clases_np for pred in predictions)
                success_rate = successful_attacks / len(predictions)
                epoch_success_rates.append(success_rate)
        
        # Epoch stats
        avg_epoch_loss = np.mean(epoch_losses)
        avg_epoch_success = np.percentile(epoch_success_rates, 90)
        
        losses.append(avg_epoch_loss)
        success_rates.append(avg_epoch_success)
        
        # Track epochs to 50% threshold
        if epochs_to_50_percent is None and avg_epoch_success >= 0.5:
            epochs_to_50_percent = epoch
        
        # Log to comet
        experiment.log_metric(f"{exp_name}_loss", avg_epoch_loss, step=epoch)
        experiment.log_metric(f"{exp_name}_success_rate", avg_epoch_success, step=epoch)
        experiment.log_metric(f"{exp_name}_aug_weight", aug_weight, step=epoch)
        
        # Per-patch evaluation (every 5 epochs)
        if epoch % 5 == 0 or avg_epoch_success > 0.3:
            torch.cuda.empty_cache()  # Clear memory before validation
            with torch.no_grad():
                all_patches = resizer(decode_latents(latent_batch).float())
                # Reduce batch limit for larger latent sizes to save memory
                val_batch_limit = min(2 if latent_size >= 16 else 3, len(val_loader))
                
                for patch_idx in range(num_patches):
                    patch_clean_successes = 0
                    patch_aug_successes = 0
                    total_tests = 0
                    single_patch = all_patches[patch_idx:patch_idx+1]
                    
                    for val_batch_idx, (val_frames, val_H_t) in enumerate(val_loader):
                        if val_batch_idx >= val_batch_limit:
                            break
                        
                        val_frames = val_frames.to(device)
                        val_H_t = val_H_t.to(device)
                        
                        # === CLEAN PATCH TEST ===
                        w_mask = warp(single_patch * 0 + 1, val_H_t)
                        w_patch = warp(single_patch, val_H_t)
                        clean_blended = ((w_mask != 0) * -BLEND_RATIO + 1) * val_frames + w_patch * BLEND_RATIO
                        clean_batch = clean_blended.view(-1, *clean_blended.shape[2:])
                        
                        clean_logits = predict_raw_dev(clean_batch)
                        clean_predictions = clean_logits.argmax(dim=1)
                        
                        # === AUGMENTED PATCH TEST ===
                        aug_patch = jitter(single_patch)
                        try:
                            aug_patch = torch.stack([augmentor(x).to(device) for x in aug_patch])
                        except:
                            pass  # Use only jitter if augmentor fails
                        w_mask_aug = warp(aug_patch * 0 + 1, val_H_t)
                        w_patch_aug = warp(aug_patch, val_H_t)
                        aug_blended = ((w_mask_aug != 0) * -BLEND_RATIO + 1) * val_frames + w_patch_aug * BLEND_RATIO
                        aug_blended = aug_blended.squeeze(0)
                        aug_blended = jitter_total_photo(aug_blended)
                        aug_batch = aug_blended.view(-1, *aug_blended.shape[1:])
                        
                        aug_logits = predict_raw_dev(aug_batch)
                        aug_predictions = aug_logits.argmax(dim=1)
                        
                        # Count successes
                        for pred in clean_predictions:
                            if pred.item() not in orig_clases_np:
                                patch_clean_successes += 1
                            total_tests += 1
                        
                        for pred in aug_predictions:
                            if pred.item() not in orig_clases_np:
                                patch_aug_successes += 1
                    
                    # Calculate success rates for this patch
                    clean_rate = patch_clean_successes / total_tests if total_tests > 0 else 0
                    aug_rate = patch_aug_successes / total_tests if total_tests > 0 else 0
                    
                    patch_success_history[patch_idx].append(clean_rate)
                    patch_augmented_success_history[patch_idx].append(aug_rate)
                
                # Report top performing patches
                if len(patch_success_history[0]) > 0:
                    print(f"🏆 TOP 5 PATCHES (Augmented Performance):")
                    
                    latest_aug_performance = [(i, patch_augmented_success_history[i][-1]) for i in range(num_patches)]
                    latest_aug_performance.sort(key=lambda x: x[1], reverse=True)
                    best_patch_idx, best_patch_rate = latest_aug_performance[0]
                    
                    for rank, (patch_idx, aug_rate_val) in enumerate(latest_aug_performance[:5], 1):
                        clean_rate = patch_success_history[patch_idx][-1]
                        robustness = (aug_rate_val / clean_rate) if clean_rate > 0 else 0
                        print(f"  {rank}. Patch #{patch_idx+1:2d}: Clean {clean_rate:.1%} | Aug {aug_rate_val:.1%} | Robust {robustness:.1%}")
                    
                    # Show worst performers too
                    print(f"📉 BOTTOM 3 PATCHES (Augmented Performance):")
                    for rank, (patch_idx, aug_rate_val) in enumerate(latest_aug_performance[-3:], 1):
                        clean_rate = patch_success_history[patch_idx][-1]
                        print(f"  {rank}. Patch #{patch_idx+1:2d}: Clean {clean_rate:.1%} | Aug {aug_rate_val:.1%}")
                    
                    aug_success_rates.append(best_patch_rate)
                    experiment.log_metric(f"{exp_name}_best_aug_rate", best_patch_rate, step=epoch)
                    
                    # Incremental aug_weight update (like original notebook)
                    if torch.mean(torch.tensor([x[1] for x in latest_aug_performance[:1]])) > 0.7:    
                        if aug_weight < 1:
                            if aug_weight < 0.7:
                                aug_weight += 0.1
                            elif aug_weight < 0.9:
                                aug_weight += 0.05
                            else:
                                aug_weight += 0.01
                            aug_weight = min(aug_weight, 1.0)  # Cap at 1.0
                            print(f"Augmentation weight increased to {aug_weight:.2f}")
                            # Update augmentor with new weight
                            augmentor = get_augmentor()
                    
                    # Check for early stopping (90% with full aug_weight)
                    if aug_weight >= 1 and best_patch_rate >= 0.9:
                        print(f"\n🎉 BREAKTHROUGH! Patch #{best_patch_idx+1} achieved {best_patch_rate:.1%} success rate!")
                        print(f"🛑 STOPPING TRAINING - 90% threshold exceeded!")
                        training_stopped = True
                    
                    # Rejuvenation logic
                    if to_rejuvenate and latest_aug_performance[0][1] > 0.2:
                        print(f"🔧 Rejuvenating weakest patches based on augmented performance...")
                        latent_batch = latent_batch.clone().detach()
                        latent_batch_best = latent_batch[[x[0] for x in latest_aug_performance[:5]]]
                        latent_batch[[x[0] for x in latest_aug_performance[-5:]]] = latent_batch_best + (torch.randn_like(latent_batch_best) * 0.1)
                        latent_batch.requires_grad = True
                        latent_opt = torch.optim.Adam([latent_batch], lr=0.1)
        
        # Check if training should stop
        if training_stopped:
            break
        
        # Update best
        if avg_epoch_success > best_success_rate:
            best_success_rate = avg_epoch_success
            best_loss = avg_epoch_loss
            best_latent = latent_batch.clone().detach()
        
        # Progress
        if epoch % 10 == 0:
            print(f"  [{exp_name}] Epoch {epoch:3d}/{num_epochs} | Loss: {avg_epoch_loss:.3f} | Success: {avg_epoch_success:.1%}")
    
    # Save results
    final_aug_rate = aug_success_rates[-1] if aug_success_rates else 0
    
    # Create experiment-specific directory with meaningful name
    exp_dir = f'./results/ablation/{exp_name}_{curr_without_sec}'
    os.makedirs(exp_dir, exist_ok=True)
    
    # Save all patches (best latent batch)
    torch.save(best_latent, f'{exp_dir}/latent_batch_best.pt')
    
    # Also save final latent batch (may differ from best if training didn't improve at end)
    # Note: latent_batch may have been deleted if training stopped early, use best_latent
    torch.save(best_latent, f'{exp_dir}/latent_batch_final.pt')
    
    # Decode and save individual patch images
    with torch.no_grad():
        if best_latent is not None:
            decoded_patches = resizer(decode_latents(best_latent).float())
            for patch_idx in range(decoded_patches.shape[0]):
                patch_img = decoded_patches[patch_idx].cpu().permute(1, 2, 0).numpy()
                patch_img = (patch_img * 255).astype(np.uint8)
                cv2.imwrite(f'{exp_dir}/patch_{patch_idx+1:02d}.png', cv2.cvtColor(patch_img, cv2.COLOR_RGB2BGR))
    
    # Save experiment metadata
    exp_metadata = {
        'config': config,
        'best_success_rate': best_success_rate,
        'final_aug_rate': final_aug_rate,
        'epochs_to_50_percent': epochs_to_50_percent,
        'total_epochs_run': len(losses),
        'success_rate_history': success_rates,
        'loss_history': losses,
        'aug_success_rates': aug_success_rates,
    }
    with open(f'{exp_dir}/metadata.pkl', 'wb') as f:
        pickle.dump(exp_metadata, f)
    
    print(f"   💾 Patches saved to: {exp_dir}/")
    
    # Store in global results
    global_results["configs"].append(config)
    global_results["success_rate_history"][exp_name] = success_rates
    global_results["loss_history"][exp_name] = losses
    global_results["best_success_rate"][exp_name] = best_success_rate
    global_results["epochs_to_threshold"][exp_name] = epochs_to_50_percent
    global_results["final_aug_rate"][exp_name] = final_aug_rate
    
    print(f"\n✅ Experiment {exp_name} Complete!")
    print(f"   Best Success Rate: {best_success_rate:.1%}")
    print(f"   Final Aug Rate: {final_aug_rate:.1%}")
    print(f"   Epochs to 50%: {epochs_to_50_percent if epochs_to_50_percent else 'Not reached'}")
    
    # Cleanup GPU memory after experiment
    import gc
    del latent_batch
    del latent_opt
    del resizer
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.empty_cache()
    print(f"   🧹 GPU memory cleared")
    
    return global_results

## Run All Ablation Experiments

In [14]:
# Run all experiments sequentially
print("🚀 Starting Ablation Study...")
print(f"Running {len(ABLATION_CONFIGS)} experiments with {NUM_EPOCHS} epochs each\n")

for i, config in enumerate(ABLATION_CONFIGS, 1):
    # Skip already completed experiments
    if config['name'] in ablation_results.get('best_success_rate', {}):
        print(f"\n[{i}/{len(ABLATION_CONFIGS)}] Skipping: {config['name']} (already completed)")
        continue
        
    print(f"\n[{i}/{len(ABLATION_CONFIGS)}] Running: {config['name']}")
    ablation_results = run_single_experiment(
        config=config,
        num_epochs=NUM_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        experiment=experiment,
        global_results=ablation_results
    )
    
    # Clear GPU memory between experiments
    torch.cuda.empty_cache()
    import gc
    gc.collect()
    torch.cuda.empty_cache()  # Double clear after gc
    print(f"💾 GPU Memory after cleanup: {torch.cuda.memory_allocated()/1024**3:.2f} GB allocated")

print("\n" + "="*60)
print("🎉 All Ablation Experiments Complete!")
print("="*60)

🚀 Starting Ablation Study...
Running 5 experiments with 30 epochs each


[1/5] Running: latent_4x4

🧪 Starting Experiment: latent_4x4
   Latent Size: 4x4
   Rejuvenation: Disabled



Epoch 1/30:   3%|▎         | 1/31 [00:01<00:53,  1.79s/it]/home/dcor/niskhizov/PhysicalAdverserialProj/interp_comp_torch.py:179: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at /pytorch/aten/src/ATen/native/BucketizationUtils.h:32.)
  indices = torch.searchsorted(x_vals, xi_expanded, right=False)
/home/dcor/niskhizov/PhysicalAdverserialProj/interp_comp_torch.py:179: UserWarning: torch.searchsorted(): boundary tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous boundary tensor if possible. This message will only appear once per program. (Triggered internally at /pytorch/aten/src/ATen/native/BucketizationUtils.h:38.)
  indices = t

🏆 TOP 5 PATCHES (Augmented Performance):
  1. Patch # 1: Clean 16.7% | Aug 3.3% | Robust 20.0%
  2. Patch # 2: Clean 10.0% | Aug 3.3% | Robust 33.3%
  3. Patch # 7: Clean 6.7% | Aug 3.3% | Robust 50.0%
  4. Patch # 8: Clean 13.3% | Aug 3.3% | Robust 25.0%
  5. Patch # 3: Clean 0.0% | Aug 0.0% | Robust 0.0%
📉 BOTTOM 3 PATCHES (Augmented Performance):
  1. Patch # 6: Clean 3.3% | Aug 0.0%
  2. Patch # 9: Clean 0.0% | Aug 0.0%
  3. Patch #10: Clean 0.0% | Aug 0.0%
  [latent_4x4] Epoch   0/30 | Loss: -27.768 | Success: 0.0%


🏆 TOP 5 PATCHES (Augmented Performance):
  1. Patch # 8: Clean 96.7% | Aug 90.0% | Robust 93.1%
  2. Patch # 9: Clean 83.3% | Aug 76.7% | Robust 92.0%
  3. Patch # 1: Clean 30.0% | Aug 20.0% | Robust 66.7%
  4. Patch # 2: Clean 20.0% | Aug 20.0% | Robust 100.0%
  5. Patch # 3: Clean 50.0% | Aug 20.0% | Robust 40.0%
📉 BOTTOM 3 PATCHES (Augmented Performance):
  1. Patch # 7: Clean 10.0% | Aug 6.7%
  2. Patch #10: Clean 6.7% | Aug 6.7%
  3. Patch # 6: Clean 6.7% | Aug 3.3%
Augmentation weight increased to 0.91


🏆 TOP 5 PATCHES (Augmented Performance):
  1. Patch # 8: Clean 100.0% | Aug 80.0% | Robust 80.0%
  2. Patch # 9: Clean 90.0% | Aug 76.7% | Robust 85.2%
  3. Patch # 4: Clean 10.0% | Aug 30.0% | Robust 300.0%
  4. Patch # 1: Clean 40.0% | Aug 23.3% | Robust 58.3%
  5. Patch # 3: Clean 50.0% | Aug 23.3% | Robust 46.7%
📉 BOTTOM 3 PATCHES (Augmented Performance):
  1. Patch # 6: Clean 10.0% | Aug 6.7%
  2. Patch # 7: Clean 10.0% | Aug 6.7%
  3. Patch #10: Clean 6.7% | Aug 3.3%
Augmentation weight increased to 0.92


...

🏆 TOP 5 PATCHES (Augmented Performance):
  1. Patch # 1: Clean 100.0% | Aug 96.7% | Robust 96.7%
  2. Patch # 4: Clean 100.0% | Aug 96.7% | Robust 96.7%
  3. Patch # 5: Clean 100.0% | Aug 93.3% | Robust 93.3%
  4. Patch # 6: Clean 100.0% | Aug 93.3% | Robust 93.3%
  5. Patch #10: Clean 100.0% | Aug 93.3% | Robust 93.3%
📉 BOTTOM 3 PATCHES (Augmented Performance):
  1. Patch # 7: Clean 63.3% | Aug 26.7%
  2. Patch # 9: Clean 56.7% | Aug 23.3%
  3. Patch # 8: Clean 60.0% | Aug 20.0%
Augmentation weight increased to 0.97


🏆 TOP 5 PATCHES (Augmented Performance):
  1. Patch # 4: Clean 100.0% | Aug 96.7% | Robust 96.7%
  2. Patch # 5: Clean 100.0% | Aug 96.7% | Robust 96.7%
  3. Patch # 1: Clean 100.0% | Aug 93.3% | Robust 93.3%
  4. Patch # 6: Clean 100.0% | Aug 93.3% | Robust 93.3%
  5. Patch #10: Clean 100.0% | Aug 93.3% | Robust 93.3%
📉 BOTTOM 3 PATCHES (Augmented Performance):
  1. Patch # 7: Clean 60.0% | Aug 30.0%
  2. Patch # 9: Clean 56.7% | Aug 23.3%
  3. Patch # 8: Clean 63.3% | Aug 20.0%
Augmentation weight increased to 0.98


🏆 TOP 5 PATCHES (Augmented Performance):
  1. Patch # 1: Clean 100.0% | Aug 96.7% | Robust 96.7%
  2. Patch # 4: Clean 100.0% | Aug 96.7% | Robust 96.7%
  3. Patch # 5: Clean 100.0% | Aug 96.7% | Robust 96.7%
  4. Patch # 6: Clean 100.0% | Aug 96.7% | Robust 96.7%
  5. Patch #10: Clean 100.0% | Aug 93.3% | Robust 93.3%
📉 BOTTOM 3 PATCHES (Augmented Performance):
  1. Patch # 7: Clean 70.0% | Aug 26.7%
  2. Patch # 9: Clean 63.3% | Aug 26.7%
  3. Patch # 8: Clean 63.3% | Aug 23.3%
Augmentation weight increased to 0.99
  [latent_8x8] Epoch  10/30 | Loss: -28.510 | Success: 84.0%


🏆 TOP 5 PATCHES (Augmented Performance):
  1. Patch # 1: Clean 100.0% | Aug 96.7% | Robust 96.7%
  2. Patch # 4: Clean 100.0% | Aug 96.7% | Robust 96.7%
  3. Patch # 5: Clean 100.0% | Aug 96.7% | Robust 96.7%
  4. Patch # 6: Clean 100.0% | Aug 96.7% | Robust 96.7%
  5. Patch #10: Clean 100.0% | Aug 96.7% | Robust 96.7%
📉 BOTTOM 3 PATCHES (Augmented Performance):
  1. Patch # 7: Clean 70.0% | Aug 23.3%
  2. Patch # 8: Clean 60.0% | Aug 23.3%
  3. Patch # 9: Clean 66.7% | Aug 23.3%
Augmentation weight increased to 1.00

🎉 BREAKTHROUGH! Patch #1 achieved 96.7% success rate!
🛑 STOPPING TRAINING - 90% threshold exceeded!
   💾 Patches saved to: ./results/ablation/latent_8x8_2026-01-17_21_28/

✅ Experiment latent_8x8 Complete!
   Best Success Rate: 84.0%
   Final Aug Rate: 96.7%
   Epochs to 50%: 3
   🧹 GPU memory cleared
💾 GPU Memory after cleanup: 2.79 GB allocated

[3/5] Running: latent_16x16

🧪 Starting Experiment: latent_16x16
   Latent Size: 16x16
   Rejuvenation: Disabled



OutOfMemoryError: CUDA out of memory. Tried to allocate 38.00 MiB. GPU 0 has a total capacity of 31.73 GiB of which 16.44 MiB is free. Including non-PyTorch memory, this process has 31.71 GiB memory in use. Of the allocated memory 31.04 GiB is allocated by PyTorch, and 262.27 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Results Analysis and Visualization

In [ ]:
# Plot success rate convergence for all experiments
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Success rate over epochs (all experiments)
ax1 = axes[0, 0]
for exp_name, success_history in ablation_results["success_rate_history"].items():
    ax1.plot(success_history, label=exp_name, linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Success Rate')
ax1.set_title('Success Rate Convergence - All Experiments')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Plot 2: Loss over epochs (all experiments)
ax2 = axes[0, 1]
for exp_name, loss_history in ablation_results["loss_history"].items():
    ax2.plot(loss_history, label=exp_name, linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Loss Convergence - All Experiments')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

# Plot 3: Best success rate comparison (bar chart)
ax3 = axes[1, 0]
exp_names = list(ablation_results["best_success_rate"].keys())
best_rates = [ablation_results["best_success_rate"][name] for name in exp_names]
bars = ax3.bar(exp_names, best_rates, color=['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12'])
ax3.set_ylabel('Best Success Rate')
ax3.set_title('Best Success Rate Comparison')
ax3.set_ylim(0, 1)
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')
for bar, rate in zip(bars, best_rates):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{rate:.1%}', 
             ha='center', va='bottom', fontsize=10)

# Plot 4: Epochs to 50% threshold
ax4 = axes[1, 1]
epochs_to_thresh = [ablation_results["epochs_to_threshold"].get(name, NUM_EPOCHS) if ablation_results["epochs_to_threshold"].get(name) else NUM_EPOCHS for name in exp_names]
bars = ax4.bar(exp_names, epochs_to_thresh, color=['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12'])
ax4.set_ylabel('Epochs to 50% Success')
ax4.set_title('Convergence Speed (Epochs to 50% Threshold)')
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
for bar, epochs in zip(bars, epochs_to_thresh):
    label = str(epochs) if epochs < NUM_EPOCHS else 'N/R'
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, label, 
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(f'./results/ablation/ablation_results_{curr_without_sec}.png', dpi=150, bbox_inches='tight')
plt.savefig(f'./results/ablation/ablation_results_{curr_without_sec}.svg', bbox_inches='tight')
experiment.log_figure(figure=fig, figure_name="ablation_results")
plt.show()

In [ ]:
# Compare latent sizes specifically
latent_size_results = {k: v for k, v in ablation_results["success_rate_history"].items() 
                       if 'rejuv' not in k or 'no_rejuv' in k}

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'latent_4x4': '#e74c3c', 'latent_8x8': '#f39c12', 'latent_16x16_no_rejuv': '#2ecc71', 'latent_32x32': '#3498db'}

for exp_name, success_history in latent_size_results.items():
    ax.plot(success_history, label=exp_name.replace('_no_rejuv', ''), linewidth=2.5, color=colors.get(exp_name, 'gray'))

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Success Rate', fontsize=12)
ax.set_title('Effect of Latent Size on Attack Success Rate', fontsize=14)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(f'./results/ablation/latent_size_comparison_{curr_without_sec}.png', dpi=150, bbox_inches='tight')
plt.savefig(f'./results/ablation/latent_size_comparison_{curr_without_sec}.svg', bbox_inches='tight')
experiment.log_figure(figure=fig, figure_name="latent_size_comparison")
plt.show()

In [ ]:
# Compare rejuvenation vs no rejuvenation
rejuv_results = {k: v for k, v in ablation_results["success_rate_history"].items() 
                 if '16x16' in k}

fig, ax = plt.subplots(figsize=(10, 6))

for exp_name, success_history in rejuv_results.items():
    label = "With Rejuvenation" if 'with_rejuv' in exp_name else "Without Rejuvenation"
    color = '#2ecc71' if 'with_rejuv' in exp_name else '#e74c3c'
    ax.plot(success_history, label=label, linewidth=2.5, color=color)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Success Rate', fontsize=12)
ax.set_title('Effect of Patch Rejuvenation on Convergence (16x16 Latent)', fontsize=14)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(f'./results/ablation/rejuvenation_comparison_{curr_without_sec}.png', dpi=150, bbox_inches='tight')
plt.savefig(f'./results/ablation/rejuvenation_comparison_{curr_without_sec}.svg', bbox_inches='tight')
experiment.log_figure(figure=fig, figure_name="rejuvenation_comparison")
plt.show()

In [ ]:
# Summary table
import pandas as pd

summary_data = []
for cfg in ablation_results["configs"]:
    name = cfg["name"]
    summary_data.append({
        "Experiment": name,
        "Latent Size": f"{cfg['latent_size']}x{cfg['latent_size']}",
        "Rejuvenation": "Yes" if cfg['rejuvenate'] else "No",
        "Best Success Rate": f"{ablation_results['best_success_rate'][name]:.1%}",
        "Final Aug Rate": f"{ablation_results['final_aug_rate'][name]:.1%}",
        "Epochs to 50%": ablation_results['epochs_to_threshold'][name] if ablation_results['epochs_to_threshold'][name] else "N/R"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("📊 ABLATION STUDY SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

# Save summary
summary_df.to_csv(f'./results/ablation/ablation_summary_{curr_without_sec}.csv', index=False)
experiment.log_table(f"ablation_summary_{curr_without_sec}.csv", summary_df)

In [ ]:
# Save all results
import pickle

with open(f'./results/ablation/ablation_results_{curr_without_sec}.pkl', 'wb') as f:
    pickle.dump(ablation_results, f)

experiment.log_asset(f'./results/ablation/ablation_results_{curr_without_sec}.pkl')
print(f"Results saved to ./results/ablation/ablation_results_{curr_without_sec}.pkl")

In [ ]:
# End experiment
experiment.end()
print("Experiment ended successfully!")